# Encapsulación y `property` en Python

La **encapsulación es un concepto clave de la programación orientada a objetos**.

La idea consiste en distinguir entre la **interfaz pública** de un objeto —lo que otros objetos y funciones deberían utilizar— y sus **detalles internos de implementación**.

En algunos lenguajes existen modificadores estrictos como `public`, `protected` y `private`. En Python, el enfoque es diferente: se basa mucho más en **convenciones entre programadores** que en restricciones absolutas del lenguaje.

## 1. Atributos públicos

Un atributo público puede accederse libremente desde fuera del objeto.

En Python, cualquier atributo cuyo nombre no empiece por guion bajo se considera público por convención.

In [ ]:
class Persona:
    def __init__(self, nombre):
        self.nombre = nombre

p = Persona("Ana")
print(p.nombre)
p.nombre = "Marta"
print(p.nombre)

Esto es completamente normal en Python.

No es necesario crear automáticamente métodos `get_nombre()` y `set_nombre()` para cada atributo, como suele hacerse en otros lenguajes.

## 2. Atributos llamados “protected”: `_atributo`

Cuando un nombre comienza por un único guion bajo:

```python
_atributo
```

Python transmite una intención:

> Este atributo está pensado para uso interno de la clase o de sus subclases.

Pero **Python no impide acceder a él desde fuera**.

In [ ]:
class Persona:
    def __init__(self, nombre):
        self._nombre = nombre

p = Persona("Ana")
print(p._nombre)   # Es posible
p._nombre = "Luis"
print(p._nombre)

Por tanto, `_nombre` no está realmente protegido.

El guion bajo es una **convención**: indica al resto de programadores que no deberían depender directamente de ese atributo salvo que exista una buena razón.

A veces se denomina atributo *protected* por analogía con otros lenguajes, pero Python no aplica una protección estricta.

## 3. Atributos con doble guion bajo: `__atributo`

Si un atributo comienza con dos guiones bajos:

```python
__atributo
```

Python aplica un mecanismo llamado **name mangling**.

A veces se habla de estos atributos como "privados", pero conviene entender que **no son privados en sentido estricto**.

In [ ]:
class Persona:
    def __init__(self, nombre):
        self.__nombre = nombre

p = Persona("Ana")

# Esto produce AttributeError:
# print(p.__nombre)

Puede parecer que `__nombre` es realmente privado, pero Python no lo oculta completamente.

Internamente, Python transforma el nombre para reducir colisiones accidentales, especialmente en escenarios de herencia.

In [ ]:
class Persona:
    def __init__(self, nombre):
        self.__nombre = nombre

p = Persona("Ana")
print(p._Persona__nombre)

Por tanto, un atributo con `__` **sigue siendo accesible** si conocemos el nombre generado.

El objetivo principal de `__atributo` no es proporcionar seguridad ni impedir por completo el acceso, sino **reducir accesos accidentales y evitar conflictos de nombres**.

## 4. ¿Existen realmente `public`, `protected` y `private`?

En Python podemos resumirlo así:

| Convención | Ejemplo | Significado habitual |
|---|---|---|
| Público | `nombre` | Forma parte de la interfaz normal |
| Uso interno | `_nombre` | No debería usarse directamente desde fuera salvo que exista una buena razón |
| Name mangling | `__nombre` | Reduce accesos accidentales y conflictos de nombres |

Por tanto:

- `_atributo` **no está protegido** por el lenguaje;
- `__atributo` **no es privado en sentido estricto**;
- Python confía en gran medida en que el programador respete la interfaz diseñada por la clase.

## 5. ¿Por qué encapsular entonces?

Aunque Python permita acceder a casi todo, la encapsulación sigue siendo útil.

Nos permite:

- separar la interfaz pública de los detalles internos;
- validar valores antes de modificar el estado;
- mantener invariantes del objeto;
- cambiar la implementación interna sin obligar a modificar el código cliente;
- hacer que el código sea más fácil de entender y mantener.

## 6. ¿Cómo controlamos entonces el acceso o la modificación?

Si Python no impide estrictamente acceder a los atributos, puede surgir una pregunta natural:

> ¿Cómo podemos controlar qué valores se asignan o qué ocurre cuando se consulta un atributo?

Aquí es donde `property` resulta especialmente útil.

Una `property` **no convierte un atributo en privado**. Su finalidad principal es mantener una sintaxis natural de atributo mientras añadimos comportamiento, por ejemplo:

- validar valores;
- calcular un valor dinámicamente;
- impedir determinadas asignaciones;
- cambiar la implementación interna sin modificar el código que usa la clase.

## 7. Getters y setters tradicionales

Podríamos escribir una clase así:

In [ ]:
class Persona:
    def __init__(self, edad):
        self._edad = edad

    def get_edad(self):
        return self._edad

    def set_edad(self, edad):
        if edad < 0:
            raise ValueError("La edad no puede ser negativa")
        self._edad = edad

Este estilo funciona, pero en Python suele resultar innecesariamente verboso.

Python ofrece `property`, que permite controlar el acceso manteniendo una sintaxis natural de atributo.

## 8. `@property`

`property` permite definir un método que, desde fuera de la clase, se utiliza con **sintaxis de atributo**.

Esto resulta útil cuando queremos mantener una interfaz sencilla como:

```python
p.edad
```

pero necesitamos ejecutar código cada vez que se consulta ese valor.

In [ ]:
class Persona:
    def __init__(self, edad):
        self._edad = edad

    @property
    def edad(self):
        return self._edad

p = Persona(20)
print(p.edad)

Aunque `edad` está implementado mediante un método, desde fuera se utiliza como un atributo:

```python
p.edad
```

y no como:

```python
p.edad()
```

## 9. Setter de una propiedad

Podemos controlar también las asignaciones utilizando `@nombre.setter`.

In [ ]:
class Persona:
    def __init__(self, edad):
        self._edad = 0
        self.edad = edad

    @property
    def edad(self):
        return self._edad

    @edad.setter
    def edad(self, valor):
        if valor < 0:
            raise ValueError("La edad no puede ser negativa")
        self._edad = valor

p = Persona(20)
print(p.edad)
p.edad = 25
print(p.edad)
# p.edad = -5  # produciría ValueError

Aquí `edad` forma parte de la **interfaz pública**, mientras que `_edad` es un **detalle interno de implementación**.

La propiedad no hace que `_edad` sea inaccesible. Lo importante es que la clase establece que la forma normal de interactuar con ese dato es mediante `edad`.

El setter permite validar el valor sin obligar al usuario de la clase a llamar a un método como `set_edad()`.

## 10. Propiedades de solo lectura

Si definimos `@property` pero no un setter, la propiedad no puede asignarse directamente.

In [ ]:
class Circulo:
    def __init__(self, radio):
        self.radio = radio

    @property
    def area(self):
        return 3.14159 * self.radio ** 2

c = Circulo(2)
print(c.area)
# c.area = 10  # produciría AttributeError

En este caso `area` parece un atributo, pero realmente se calcula cada vez que se consulta.

## 11. Una ventaja importante de `property`

Podemos empezar con un atributo público sencillo:

In [ ]:
class Persona:
    def __init__(self, edad):
        self.edad = edad

Y más adelante, si necesitamos validación o algún comportamiento adicional, podemos convertirlo en una `property` **sin cambiar la forma en que el resto del programa lo utiliza**:

```python
p.edad
p.edad = 30
```

Esta es una razón importante por la que en Python no suele recomendarse crear getters y setters triviales desde el principio "por si acaso".

Podemos comenzar con un atributo público y añadir una `property` solo cuando realmente sea necesaria.

## 12. Encapsulación en Python: idea práctica

En Python, encapsular no significa necesariamente **impedir físicamente el acceso** a los datos.

Significa principalmente:

- definir una interfaz pública clara;
- distinguir esa interfaz de los detalles internos de implementación;
- controlar modificaciones cuando sea necesario;
- mantener invariantes del objeto;
- permitir que la implementación interna cambie sin afectar al código cliente.

La encapsulación en Python depende tanto de las herramientas del lenguaje como de que los programadores respeten las convenciones y la interfaz diseñada por la clase.

## 13. Resumen

- `atributo` → público por convención.
- `_atributo` → indica uso interno, pero sigue siendo accesible.
- `__atributo` → aplica *name mangling*; no proporciona privacidad estricta.
- Python se apoya más en convenciones y diseño de interfaces que en restricciones rígidas de acceso.
- `@property` **no hace privado un atributo**: permite añadir comportamiento manteniendo sintaxis de atributo.
- `@propiedad.setter` permite validar o controlar asignaciones.
- Una propiedad sin setter puede actuar como atributo de solo lectura.
- No suele ser necesario crear getters y setters triviales desde el principio.
- La encapsulación consiste sobre todo en separar una interfaz pública estable de los detalles internos de implementación.

# Ejercicios

## Ejercicio 1

Implementa una clase `CuentaBancaria` con un atributo interno `_saldo` y una propiedad `saldo` de solo lectura.

In [ ]:
class CuentaBancaria:
    # TU CÓDIGO AQUÍ
    pass

## Ejercicio 2

Implementa una clase `Producto` con una propiedad `precio`.

El precio debe ser siempre mayor o igual que `0`. Si se intenta asignar un valor negativo, debe generarse `ValueError`.

In [ ]:
class Producto:
    # TU CÓDIGO AQUÍ
    pass

## Ejercicio 3

Implementa una clase `Temperatura` con una propiedad `celsius` y una propiedad de solo lectura `fahrenheit`.

La conversión es:

```python
fahrenheit = celsius * 9 / 5 + 32
```

In [ ]:
class Temperatura:
    # TU CÓDIGO AQUÍ
    pass